In [ ]:
直接自己编写的代码：

通过：
pred_instances = result.pred_instances
masks = pred_instances.masks.cpu().numpy()
可以得到预测的掩膜，掩膜的格式如下(由 True 和 False 组成的 np 数组)：
mask = np.array([
    [False, False, False, True, True, False, False, False],
    [False, False, False, True, True, False, False, False],
    [False, False, False, False, True, True, False, False],
    [False, False, False, False, False, True, True, False],
    [False, False, False, False, False, False, False, False],
    [False, False, False, False, False, False, False, False],
    [False, False, False, False, False, False, False, False],
    [False, False, False, False, False, False, False, False]
])
然后通过 m = np.where(m, 255, 0).astype(np.uint8) 转换为 CV_8UC1 类型，这样就可以通过 OpenCV 提取轮廓框。        

旋转最小外接矩形框：minAreaRect

In [ ]:
from mmdet.apis import init_detector, inference_detector
import mmcv
import cv2
import numpy as np
import os

# 定义模型配置文件和训练好的权重路径
config_file = '/App/cbw/mmdetection-main/configs/mask_rcnn/mask-rcnn_r50-caffe_fpn_ms-poly-3x_hualu.py'
# checkpoint_file = 'work_dirs/mask_rcnn_r50_fpn_1x_coco/latest.pth'  # 替换为你的模型路径
checkpoint_file = '/App/cbw/mmdetection-main/checkpoints/epoch_12.pth'  # 替换为你的模型路径

# 初始化模型
model = init_detector(config_file, checkpoint_file, device='cuda:0')  # GPU模式

folderPath = '/App/cbw/Datasets/UnderWaterFish/Fish_hualu_coco/test/JPEGImages/'
outputPath = '/App/cbw/mmdetection-main/output_images/'

os.makedirs(outputPath, exist_ok=True)  # 确保输出目录存在

for filename in os.listdir(folderPath):
    if filename.lower().endswith(('.jpg')):  # 检查扩展名
        image_path = os.path.join(folderPath, filename)
        # 读取图片：例如用 PIL 或 OpenCV
        print(image_path)

        # 读取图像并推理
        # image_path = '/App/cbw/Datasets/UnderWaterFish/Fish_hualu_coco/test/JPEGImages/60_2.jpg' # 1条鱼
        # image_path = '/App/cbw/Datasets/UnderWaterFish/Fish_hualu_coco/test/JPEGImages/180_2.jpg' # 2条鱼
        image = mmcv.imread(image_path)
        result = inference_detector(model, image)

        pred_instances = result.pred_instances
        scores = pred_instances.scores.cpu().numpy()
        for i, score in enumerate(scores):
            print(f"检测到第 {i+1} 个实例，分数: {scores[i]:.2f}")
        # score = np.mean(scores)  # 计算平均分数
        # print(f"平均分数: {score:.2f}")
        masks = pred_instances.masks.cpu().numpy()
        # print("掩膜信息:", masks)

        # height, width = image.shape[:2]

        # print(f"Image height: {height} pixels")
        # print(f"Image width: {width} pixels")

        # 计算每行中连续 True 值的长度
        row_length = []
        for i, m in enumerate(masks):
            
            # 转换为 CV_8UC1 类型
            m = np.where(m, 255, 0).astype(np.uint8)

            contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            if len(contours) == 0:
                print("未检测到任何轮廓")
            else:
                max_length = 0
                max_contour = None
                

                for contour in contours:
                    x, y, w, h = cv2.boundingRect(contour)
                    length = max(w, h)  # 计算最长边
                    length = max(w, h)
                    width = min(w, h)  # 计算宽
                    # 计算最小外接矩形
                    rect = cv2.minAreaRect(contour)
                    # 获取矩形的四个顶点坐标
                    box = cv2.boxPoints(rect)
                    # 将坐标转换为整数
                    box = np.int0(box)

                    # 计算矩形的中心点
                    center = rect[0]

                    # 计算矩形的宽度和高度
                    width = rect[1][0]
                    height = rect[1][1]
                    fishlength = max(width, height)  # 计算最长边
                    fishheight = min(width, height)  # 计算最短边
                    print(f"width: {fishlength}, height: {fishheight}")

                    # 在图像上绘制最小外接矩形
                    cv2.drawContours(image, [box], 0, (0, 255, 0), 2)

                    # 计算并显示矩形的中心点
                    cv2.circle(image, (int(center[0]), int(center[1])), 5, (0, 0, 255), -1)

                    # 计算并显示矩形的旋转角度
                    angle = rect[2]
                    # cv2.putText(image, f'旋转角度: {angle:.2f}°', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

                    # # 将角度转换为弧度
                    # angle_rad = np.radians(angle)  # 用于三角函数计算
                    
                    # # 计算宽度方向线条的端点（绿色线）
                    # # 宽度方向：沿角度 angle 的方向
                    # dx_width = (width / 2) * np.cos(angle_rad)  # x 方向偏移量
                    # dy_width = (width / 2) * np.sin(angle_rad)  # y 方向偏移量
                    # start_width = (int(center[0] - dx_width), int(center[1] - dy_width))
                    # end_width = (int(center[0] + dx_width), int(center[1] + dy_width))
                    # cv2.line(image, start_width, end_width, (0, 255, 0), 2)  # 绘制宽度线
                    
                    # # 计算高度方向线条的端点（红色线）
                    # # 高度方向：垂直于宽度方向（角度 angle + 90°）
                    # dx_height = (height / 2) * (-np.sin(angle_rad))  # x 方向偏移量（cos(θ+90°) = -sin(θ)）[[15]]
                    # dy_height = (height / 2) * np.cos(angle_rad)      # y 方向偏移量（sin(θ+90°) = cos(θ)）[[15]]
                    # start_height = (int(center[0] - dx_height), int(center[1] - dy_height))
                    # end_height = (int(center[0] + dx_height), int(center[1] + dy_height))
                    # cv2.line(image, start_height, end_height, (0, 0, 255), 2)  # 绘制高度线

                    # 计算并显示矩形的宽度和高度
                    # cv2.putText(image, f'宽度: {width}, 高度: {height}', (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)



                #     cv2.putText(image, f'宽度: {width}, 高度: {height}', (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                
                    if length > max_length:
                        max_length = length
                        max_contour = contour

                    # 在原始图像上绘制最大轮廓
                    cv2.drawContours(image, [max_contour], 0, (0, 255, 0), 2)

                    # cv2.putText(image, f'length: {f"{fishlength:.2f}"}', (x + 10, y + 20), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                    # cv2.putText(image, f'height: {f"{fishheight:.2f}"}', (x + 10, y + 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)
                    # 置信度显示不是这样的吧。。。。。。 有的多了。。。
                    # cv2.putText(image, f'score:{f"{scores[i]:.2f}"}', (x + 10, y + 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)
            
            outputFile = os.path.join(outputPath, filename)
            cv2.imwrite(outputFile, image) 

In [ ]:
# 体长
体长 = [17.07, 17.39, 17.89, 18.09, 18.13, 16.75, 16.74, 16.97, 17.19, 18.08, 16.84, 16.73, 17.08, 16.91, 18.15, 17.02]

# 体高
体高 = [5.72, 5.89, 6.51, 6.62, 6.39, 5.94, 5.72, 5.78, 5.76, 6.47, 5.89, 5.92, 5.77, 5.91, 6.42, 5.72]

# 全长
全长 = [20.29, 20.45, 21.6, 21.7, 21.78, 20.01, 20, 20.01, 20.25, 21.61, 20.18, 20.25, 20.18, 20.05, 21.79, 20.22]

# 吻长
吻长 = [1.53, 1.71, 1.71, 1.64, 1.6, 2.04, 1.49, 1.64, 1.75, 1.49, 2, 1.82, 1.79, 2.11, 1.6, 1.71]

# 头长
头长 = [6.43, 6.65, 6.33, 6.61, 6.36, 6.68, 6.14, 6.05, 6.24, 6.3, 6.4, 6.65, 6.49, 6.74, 6.33, 6.4]

# 尾柄长
尾柄长 = [3.31, 3.8, 3.46, 3.22, 3.57, 2.91, 3.08, 2.97, 3.72, 3.01, 3.12, 3.07, 3.06, 3.21, 3.74, 3.22]

# 尾柄高
尾柄高 = [1.86, 1.84, 2.15, 2.12, 2.34, 1.82, 1.83, 1.81, 1.89, 2.05, 1.87, 1.73, 1.75, 1.8, 2.32, 1.79]

# 眼后头长
眼后头长 = [3.47, 3.32, 3.16, 3.65, 3.47, 3.1, 3.13, 2.92, 3.01, 3.38, 2.95, 3.22, 3.19, 3.19, 3.35, 3.25]

# 眼径
眼径 = [1.3, 1.49, 1.35, 1.22, 1.2, 1.46, 1.38, 1.38, 1.38, 1.3, 1.38, 1.49, 1.41, 1.38, 1.27, 1.33]

19.6	6.8  	24.2	1.9	 7.5	 3.5	2.3 	3.5 	1.4

In [32]:
measurements = [1.3, 1.49, 1.35, 1.22, 1.2, 1.46, 1.38, 1.38, 1.38, 1.3, 1.38, 1.49, 1.41, 1.38, 1.27, 1.33]
mse = sum((1.4	 - x)**2 for x in measurements) / 16
print(f"MSE: {mse:.4f}")

MSE: 0.0087


In [12]:
measurements = [7.19, 7.33, 7.19, 7.08, 7.06, 7.39, 7.46, 7.21, 7.26, 7.2, 7.41]
mse = sum((6.8 - x)**2 for x in measurements) / 11
print(f"MSE: {mse:.4f}")

MSE: 0.2206


In [40]:
import math

number = 0.27


result = math.sqrt(number)
print(f"平方根是 {result}")

平方根是 0.5196152422706632
